# 📊 Data Quality Exploration — Great Expectations POC

This notebook walks through:
1. Loading and previewing each dataset
2. Basic profiling (describe, null counts, duplicates)
3. Clean vs dirty customer comparison
4. Running GX validations inline
5. Displaying results as formatted DataFrames

> **Prerequisites:** Run `python main.py` first to generate all datasets and set up GX.

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import get_data_raw_dir, get_data_bad_dir

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'Project root: {PROJECT_ROOT}')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

## 1. Load & Preview Datasets

In [ ]:
raw = get_data_raw_dir()
bad = get_data_bad_dir()

customers = pd.read_csv(raw / 'customers.csv')
orders = pd.read_csv(raw / 'orders.csv')
products = pd.read_csv(raw / 'products.csv')
dirty_customers = pd.read_csv(bad / 'customers_dirty.csv')

print('Dataset shapes:')
for name, df in [('customers', customers), ('orders', orders),
                  ('products', products), ('dirty_customers', dirty_customers)]:
    print(f'  {name:<20} → {df.shape[0]:>6,} rows × {df.shape[1]} cols')

In [ ]:
print('=== CUSTOMERS (first 5 rows) ===')
customers.head()

In [ ]:
print('=== ORDERS (first 5 rows) ===')
orders.head()

In [ ]:
print('=== PRODUCTS (first 5 rows) ===')
products.head()

## 2. Basic Profiling

In [ ]:
def profile_df(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Return a concise profiling summary for a DataFrame."""
    profile = pd.DataFrame({
        'dtype': df.dtypes,
        'null_count': df.isnull().sum(),
        'null_pct': (df.isnull().sum() / len(df) * 100).round(2),
        'unique_count': df.nunique(),
        'unique_pct': (df.nunique() / len(df) * 100).round(2),
    })
    print(f'\n── {name} ({'─' * (50 - len(name))})')
    print(f'   Rows: {len(df):,}  |  Cols: {len(df.columns)}')
    print(f'   Duplicated rows: {df.duplicated().sum()}')
    return profile

profile_df(customers, 'CUSTOMERS (clean)')

In [ ]:
customers.describe()

In [ ]:
profile_df(orders, 'ORDERS (clean)')

In [ ]:
profile_df(products, 'PRODUCTS (clean)')

## 3. Clean vs Dirty Customer Comparison

In [ ]:
def compare_datasets(clean: pd.DataFrame, dirty: pd.DataFrame) -> pd.DataFrame:
    """Side-by-side quality comparison between clean and dirty datasets."""
    metrics = {
        'row_count': [len(clean), len(dirty)],
        'null_email_count': [clean['email'].isnull().sum(), dirty['email'].isnull().sum()],
        'null_email_pct': [
            round(clean['email'].isnull().mean() * 100, 1),
            round(dirty['email'].isnull().mean() * 100, 1)
        ],
        'duplicate_ids': [
            clean['customer_id'].duplicated().sum(),
            dirty['customer_id'].duplicated().sum()
        ],
        'age_out_of_range': [
            (~clean['age'].between(18, 80)).sum(),
            (~dirty['age'].between(18, 80)).sum()
        ],
        'invalid_credit_score': [
            (~clean['credit_score'].between(300, 850)).sum(),
            (~dirty['credit_score'].between(300, 850)).sum()
        ],
        'malformed_emails': [
            clean['email'].dropna().str.contains('@').eq(False).sum(),
            dirty['email'].dropna().str.contains('@').eq(False).sum()
        ],
    }
    return pd.DataFrame(metrics, index=['clean', 'dirty']).T

comparison = compare_datasets(customers, dirty_customers)

# Highlight differences
def highlight_dirty(val):
    if isinstance(val, (int, float)) and val > 0:
        return 'background-color: #ffcccc; font-weight: bold'
    return ''

comparison.style.applymap(highlight_dirty, subset=['dirty'])

## 4. Run GX Validations Inline

In [ ]:
from src.gx_setup import setup_gx
from src.expectations_builder import build_all_suites
from src.checkpoint_runner import run_all_validations

print('Initialising GX context …')
context, assets = setup_gx()
build_all_suites(context)
print('Running all validations …')
results = run_all_validations(context)
print('\nDone!')

## 5. Validation Results Summary

In [ ]:
summary_rows = []
for r in results:
    summary_rows.append({
        'Dataset': r['dataset'],
        'Suite': r['suite'],
        'Evaluated': r['evaluated'],
        'Passed': r['successful'],
        'Failed': r['unsuccessful'],
        'Success %': r['success_pct'],
        'Status': '✅ PASS' if r['success'] else '❌ FAIL',
    })

summary_df = pd.DataFrame(summary_rows)

def highlight_status(row):
    if row['Status'] == '✅ PASS':
        return ['background-color: #ccffcc'] * len(row)
    return ['background-color: #ffcccc'] * len(row)

summary_df.style.apply(highlight_status, axis=1)

In [ ]:
# Print the path to open Data Docs
from src.utils import get_gx_context_dir

docs_index = get_gx_context_dir() / 'uncommitted' / 'data_docs' / 'local_site' / 'index.html'
print(f'📊 Data Docs: file://{docs_index.resolve()}')
print(f'   Run in terminal: xdg-open "{docs_index.resolve()}"')